# PaperMind v4 — Drive parquet → Supabase **PaperCard + GhostCard** upload

**Migration**: `db/migrations/0003_paper_anchor_facts.sql`  
**Hedef tablolar**: `fact_paper_id_card` (24.87M) + `dim_ghost_paper` (31.85M)  
**Tier**: Supabase Pro + Compute add-on **2XL** (8 vCPU / 32 GB RAM) — upload süresince  
**Beklenen süre**: PaperCard ~30-45 dk + GhostCard ~45-60 dk = **toplam ~1.5-2 saat**  
**Drive yolları (ENVANTER §10.1 A-evidence)**:
- `~/Dataleak/facts/fact_paper_id_card.parquet` (24,866,945 × 15, v1.2 incl is_suspicious)
- `~/Dataleak/dims/dim_ghost_paper.parquet` (31,855,437 × 16, v1.2 incl triage_priority)

**Bağlanti**: Session Pooler URL zorunlu (Direct IPv6, Colab IPv4 → uyumsuz).  
**Çalıştırma sırası**: Cell 1 → 2 → 3 → 4 → 5 → 6. Her cell elle çalıştır (Run all yasak — B42 hata payı).

## Cell 1 — Setup (Drive + install + DB connection)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

!pip -q install psycopg2-binary==2.9.9 pandas==2.2.2 pyarrow==17.0.0

import os, json, math, time, sys
import pandas as pd
import pyarrow.parquet as pq
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime, timedelta

DB_URL = userdata.get('SUPABASE_DB_URL')
assert DB_URL and DB_URL.startswith('postgresql://'), 'Colab Secrets → SUPABASE_DB_URL (Session Pooler)'

DRIVE_ROOT = '/content/drive/MyDrive/Dataleak'
PATHS = {
    'fact_paper_id_card': f'{DRIVE_ROOT}/facts/fact_paper_id_card.parquet',
    'dim_ghost_paper':    f'{DRIVE_ROOT}/dims/dim_ghost_paper.parquet',
}

CHUNK = 50_000          # row count per insert chunk
PAGE_SIZE = 5_000       # psycopg2 execute_values internal page
BATCH_ROWS = 500_000    # pyarrow streaming batch (RAM control)

for name, path in PATHS.items():
    exists = os.path.exists(path)
    size_gb = os.path.getsize(path) / 1e9 if exists else 0
    print(f"  {'✓' if exists else '✗'}  {name:25s} {path}  ({size_gb:.2f} GB)")

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT version();')
    print('\nDB:', cur.fetchone()[0][:60])
    cur.execute('SELECT version FROM public.schema_migrations ORDER BY applied_at;')
    print('Applied migrations:', [r[0] for r in cur.fetchall()])
    cur.execute("SELECT current_setting('shared_buffers'), current_setting('work_mem');")
    print('shared_buffers / work_mem:', cur.fetchone())

## Cell 2 — Schema audit (parquet head + dtypes — envanter doğrulaması)

In [ ]:
for name, path in PATHS.items():
    pf = pq.ParquetFile(path)
    print(f'=== {name} ===')
    print(f'  total_rows: {pf.metadata.num_rows:,}')
    print(f'  row_groups: {pf.metadata.num_row_groups}')
    print(f'  columns:    {pf.schema_arrow.names}')
    head = next(pf.iter_batches(batch_size=3)).to_pandas()
    print(f'  dtypes:\n{head.dtypes}')
    print(f'  head:\n{head}\n')

## Cell 3 — Migration 0003 uygula (PaperCard + GhostCard DDL)

**NOT**: `dim_ghost_paper` 0001 init'te placeholder olarak yaratılmıştı; bu migration `DROP TABLE IF EXISTS ... CASCADE` ile warehouse-mirror schema'ya **replace** eder. enrichment_log FK varsa ON DELETE CASCADE ile düşer (isteyerek).

In [ ]:
MIGRATION_0003 = r'''
CREATE TABLE IF NOT EXISTS public.fact_paper_id_card (
  paper_id          text PRIMARY KEY,
  pmid              text NOT NULL,
  topic_profile     jsonb NOT NULL,
  language          text,
  year              int,
  is_oa             boolean,
  type_c            real,
  type_m            real,
  type_e            real,
  type_r            real,
  type_b            real,
  dominant_type     text,
  type_confidence   real,
  version_id        text NOT NULL DEFAULT 'v1.1_2026-04-29',
  is_suspicious     boolean NOT NULL DEFAULT false,
  CHECK (dominant_type IN ('C','M','E','R','B'))
);
CREATE INDEX IF NOT EXISTS idx_paper_card_pmid       ON public.fact_paper_id_card(pmid);
CREATE INDEX IF NOT EXISTS idx_paper_card_dominant   ON public.fact_paper_id_card(dominant_type);
CREATE INDEX IF NOT EXISTS idx_paper_card_lang       ON public.fact_paper_id_card(language);
CREATE INDEX IF NOT EXISTS idx_paper_card_year       ON public.fact_paper_id_card(year) WHERE year IS NOT NULL;
CREATE INDEX IF NOT EXISTS idx_paper_card_topic_gin   ON public.fact_paper_id_card USING gin (topic_profile);
CREATE INDEX IF NOT EXISTS idx_paper_card_suspicious  ON public.fact_paper_id_card(is_suspicious) WHERE is_suspicious = true;
ALTER TABLE public.fact_paper_id_card ENABLE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS paper_card_read_all       ON public.fact_paper_id_card;
DROP POLICY IF EXISTS paper_card_write_service  ON public.fact_paper_id_card;
CREATE POLICY paper_card_read_all      ON public.fact_paper_id_card FOR SELECT TO authenticated USING (true);
CREATE POLICY paper_card_write_service ON public.fact_paper_id_card FOR ALL    TO service_role  USING (true) WITH CHECK (true);

DROP TABLE IF EXISTS public.dim_ghost_paper CASCADE;
CREATE TABLE public.dim_ghost_paper (
  paper_id                       text PRIMARY KEY,
  inferred_pmid                  text NOT NULL,
  pmid_confidence_per_segment    jsonb NOT NULL,
  citer_field_distribution       jsonb NOT NULL,
  citer_language_distribution    jsonb NOT NULL,
  language_inferred              text,
  year_upper_bound               int,
  pagerank                       double precision,
  indegree                       bigint,
  outdegree                      int,
  n_corpus_citers                bigint NOT NULL,
  is_canonical                   boolean NOT NULL DEFAULT true,
  version_id                     text NOT NULL DEFAULT 'v1.1_2026-04-29',
  computed_at                    text,
  enrichment_status              text NOT NULL DEFAULT 'pending',
  triage_priority                text NOT NULL DEFAULT 'none',
  CHECK (n_corpus_citers >= 3),
  CHECK (enrichment_status IN ('pending','fetching','enriched','failed')),
  CHECK (triage_priority IN ('high','medium','low','none'))
);
CREATE INDEX idx_ghost_pmid          ON public.dim_ghost_paper(inferred_pmid);
CREATE INDEX idx_ghost_pagerank      ON public.dim_ghost_paper(pagerank DESC) WHERE pagerank IS NOT NULL;
CREATE INDEX idx_ghost_indegree      ON public.dim_ghost_paper(indegree DESC);
CREATE INDEX idx_ghost_lang          ON public.dim_ghost_paper(language_inferred);
CREATE INDEX idx_ghost_enrich_status ON public.dim_ghost_paper(enrichment_status) WHERE enrichment_status <> 'enriched';
CREATE INDEX idx_ghost_triage        ON public.dim_ghost_paper(triage_priority)  WHERE triage_priority IN ('high','medium');
ALTER TABLE public.dim_ghost_paper ENABLE ROW LEVEL SECURITY;
CREATE POLICY ghost_read_all       ON public.dim_ghost_paper FOR SELECT TO authenticated USING (true);
CREATE POLICY ghost_write_service  ON public.dim_ghost_paper FOR ALL    TO service_role  USING (true) WITH CHECK (true);

INSERT INTO public.schema_migrations (version, description)
VALUES ('0003_paper_anchor_facts',
        'fact_paper_id_card v1.2 (24.87M × 15 incl is_suspicious) + dim_ghost_paper v1.2 (31.85M × 16 disk canon: outdegree, no q_proxy, incl triage_priority, full mirror replace)')
ON CONFLICT (version) DO NOTHING;
'''

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute(MIGRATION_0003)
    conn.commit()
    cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema='public' AND table_name IN ('fact_paper_id_card','dim_ghost_paper')
    ORDER BY table_name;
    """)
    print('Created:', [r[0] for r in cur.fetchall()])
print('✓ migration 0003 applied')

## Cell 4 — Upload **PaperCard** (24.87M × 15, ~10 GB Postgres, ~30-45 dk @ 2XL)

**Strateji**: `pyarrow.ParquetFile.iter_batches(BATCH_ROWS=500K)` → her batch içinde `CHUNK=50K` ile bulk INSERT. RAM ~3-4 GB tepe, Colab Pro+ rahat.  
**Canlı takip**: her chunk'ta zaman, throughput, ETA print.  
**topic_profile**: Struct `{theme_ids: List[str], theme_scores: List[float]}` → `json.dumps(dict)` → JSONB cast.

In [ ]:
TARGET_TABLE = 'fact_paper_id_card'
TARGET_PATH  = PATHS['fact_paper_id_card']

pf = pq.ParquetFile(TARGET_PATH)
TOTAL_ROWS = pf.metadata.num_rows
print(f'TOTAL_ROWS: {TOTAL_ROWS:,}  (~{TOTAL_ROWS/CHUNK:.0f} chunks of {CHUNK:,})')

TYPE_SET = {'C','M','E','R','B'}

def _fnum(v):
    if v is None: return None
    try:
        f = float(v)
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None

def _iint(v):
    if v is None: return None
    try:
        f = float(v)
        return None if math.isnan(f) else int(f)
    except (TypeError, ValueError):
        return None

def _topic_profile_json(v):
    """Struct → dict-like → JSON string. Numpy/list/tuple normalize."""
    if v is None:
        return json.dumps({'theme_ids': [], 'theme_scores': []})
    if isinstance(v, dict):
        d = v
    elif hasattr(v, 'as_py'):
        d = v.as_py()
    else:
        d = dict(v)
    tids = list(d.get('theme_ids') or [])
    tids = [str(x) for x in tids]
    tsc = list(d.get('theme_scores') or [])
    tsc = [float(x) for x in tsc]
    return json.dumps({'theme_ids': tids, 'theme_scores': tsc})

def _norm_dom(v):
    s = str(v).strip().upper() if v is not None else ''
    return s if s in TYPE_SET else None

INSERT_SQL = '''
INSERT INTO public.fact_paper_id_card
  (paper_id, pmid, topic_profile, language, year, is_oa,
   type_c, type_m, type_e, type_r, type_b,
   dominant_type, type_confidence, version_id, is_suspicious)
VALUES %s
ON CONFLICT (paper_id) DO NOTHING;
'''

TEMPLATE = '(%s,%s,%s::jsonb,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'

t0 = time.time()
rows_done = 0
skipped_dom = 0

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    for batch in pf.iter_batches(batch_size=BATCH_ROWS):
        df = batch.to_pandas()
        rows = []
        for r in df.itertuples(index=False):
            d = _norm_dom(r.dominant_type)
            if d is None:
                skipped_dom += 1
                continue
            rows.append((
                str(r.paper_id),
                str(r.pmid),
                _topic_profile_json(r.topic_profile),
                str(r.language) if r.language is not None else None,
                _iint(r.year),
                bool(r.is_oa) if r.is_oa is not None else None,
                _fnum(r.type_c),
                _fnum(r.type_m),
                _fnum(r.type_e),
                _fnum(r.type_r),
                _fnum(r.type_b),
                d,
                _fnum(r.type_confidence),
                str(r.version_id) if r.version_id is not None else 'v1.1_2026-04-29',
                bool(getattr(r, 'is_suspicious', False)) if getattr(r, 'is_suspicious', None) is not None else False,
            ))

        for i in range(0, len(rows), CHUNK):
            execute_values(cur, INSERT_SQL, rows[i:i+CHUNK],
                           template=TEMPLATE, page_size=PAGE_SIZE)
            rows_done += min(CHUNK, len(rows) - i)
            elapsed = time.time() - t0
            rate = rows_done / elapsed if elapsed > 0 else 0
            eta_s = (TOTAL_ROWS - rows_done) / rate if rate > 0 else 0
            eta_str = str(timedelta(seconds=int(eta_s)))
            stamp = datetime.now().strftime('%H:%M:%S')
            print(f'  [{stamp}] {rows_done:>11,}/{TOTAL_ROWS:,}  '
                  f'({100*rows_done/TOTAL_ROWS:5.1f}%)  '
                  f'rate={rate:>7,.0f}/s  ETA={eta_str}', flush=True)

        conn.commit()
        del df, rows

print(f'\n✓ {TARGET_TABLE} uploaded in {timedelta(seconds=int(time.time()-t0))}  '
      f'(skipped_dom_invalid={skipped_dom:,})')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM public.fact_paper_id_card;')
    print(f'  DB count: {cur.fetchone()[0]:,}')

## Cell 5 — Upload **GhostCard** (31.85M × 16, ~15 GB Postgres, ~45-60 dk @ 2XL)

**Struct → JSONB**: 3 alan (`pmid_confidence_per_segment`, `citer_field_distribution`, `citer_language_distribution`).  
**K7**: `year_upper_bound` iç hesap, UI'da gösterilmez (RLS gereği değil — uygulama katmanı uyar).

In [ ]:
TARGET_TABLE = 'dim_ghost_paper'
TARGET_PATH  = PATHS['dim_ghost_paper']

pf = pq.ParquetFile(TARGET_PATH)
TOTAL_ROWS = pf.metadata.num_rows
print(f'TOTAL_ROWS: {TOTAL_ROWS:,}  (~{TOTAL_ROWS/CHUNK:.0f} chunks of {CHUNK:,})')

ENRICH_SET = {'pending','fetching','enriched','failed'}

def _struct_json(v):
    """Generic Struct → dict → JSON string (numpy/list/tuple normalize)."""
    if v is None:
        return json.dumps({})
    if hasattr(v, 'as_py'):
        v = v.as_py()
    if isinstance(v, dict):
        d = {}
        for k, vv in v.items():
            if vv is None:
                d[str(k)] = None
            elif hasattr(vv, 'tolist'):
                d[str(k)] = vv.tolist()
            elif isinstance(vv, (list, tuple)):
                d[str(k)] = list(vv)
            else:
                d[str(k)] = vv
        return json.dumps(d, default=str)
    return json.dumps(v, default=str)


def _norm_triage(v):
    if v is None: return 'none'
    s = str(v).strip().lower()
    return s if s in {'high','medium','low','none'} else 'none'

def _norm_enrich(v):
    if v is None: return 'pending'
    s = str(v).strip().lower()
    return s if s in ENRICH_SET else 'pending'

INSERT_SQL = '''
INSERT INTO public.dim_ghost_paper
  (paper_id, inferred_pmid,
   pmid_confidence_per_segment, citer_field_distribution, citer_language_distribution,
   language_inferred, year_upper_bound, pagerank, indegree, n_corpus_citers,
   outdegree, is_canonical, version_id, computed_at, enrichment_status, triage_priority)
VALUES %s
ON CONFLICT (paper_id) DO NOTHING;
'''

TEMPLATE = ('(%s,%s,%s::jsonb,%s::jsonb,%s::jsonb,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)')

t0 = time.time()
rows_done = 0
skipped_n = 0

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    for batch in pf.iter_batches(batch_size=BATCH_ROWS):
        df = batch.to_pandas()
        rows = []
        for r in df.itertuples(index=False):
            n_citers = _iint(r.n_corpus_citers)
            if n_citers is None or n_citers < 3:
                skipped_n += 1
                continue
            rows.append((
                str(r.paper_id),
                str(r.inferred_pmid),
                _struct_json(r.pmid_confidence_per_segment),
                _struct_json(r.citer_field_distribution),
                _struct_json(r.citer_language_distribution),
                str(r.language_inferred) if r.language_inferred is not None else None,
                _iint(r.year_upper_bound),
                _fnum(r.pagerank),
                _iint(r.indegree),
                n_citers,
                _iint(getattr(r, 'outdegree', None)),
                bool(r.is_canonical) if r.is_canonical is not None else True,
                str(r.version_id) if r.version_id is not None else 'v1.1_2026-04-29',
                str(r.computed_at) if r.computed_at is not None else None,
                _norm_enrich(r.enrichment_status),
                _norm_triage(getattr(r, 'triage_priority', None)),
            ))

        for i in range(0, len(rows), CHUNK):
            execute_values(cur, INSERT_SQL, rows[i:i+CHUNK],
                           template=TEMPLATE, page_size=PAGE_SIZE)
            rows_done += min(CHUNK, len(rows) - i)
            elapsed = time.time() - t0
            rate = rows_done / elapsed if elapsed > 0 else 0
            eta_s = (TOTAL_ROWS - rows_done) / rate if rate > 0 else 0
            eta_str = str(timedelta(seconds=int(eta_s)))
            stamp = datetime.now().strftime('%H:%M:%S')
            print(f'  [{stamp}] {rows_done:>11,}/{TOTAL_ROWS:,}  '
                  f'({100*rows_done/TOTAL_ROWS:5.1f}%)  '
                  f'rate={rate:>7,.0f}/s  ETA={eta_str}', flush=True)

        conn.commit()
        del df, rows

print(f'\n✓ {TARGET_TABLE} uploaded in {timedelta(seconds=int(time.time()-t0))}  '
      f'(skipped_n_corpus_citers_lt_3={skipped_n:,})')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM public.dim_ghost_paper;')
    print(f'  DB count: {cur.fetchone()[0]:,}')

## Cell 6 — Verify (row counts + sample 5 PMID + dominant_type histogram + ghost top indegree)

In [ ]:
EXPECTED = {
    'fact_paper_id_card': 24_866_945,
    'dim_ghost_paper':    31_855_437,
}

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    print('=== Row counts ===')
    for tbl, exp in EXPECTED.items():
        cur.execute(f'SELECT COUNT(*) FROM public.{tbl};')
        actual = cur.fetchone()[0]
        delta = actual - exp
        flag = '✓' if abs(delta) < 1000 else ('⚠' if abs(delta) < 10000 else '✗')
        print(f'  {flag} {tbl:25s} expected={exp:>12,}  actual={actual:>12,}  Δ={delta:+,}')

    print('\n=== PaperCard: dominant_type histogram (B42-045 v1.1 hedef: C %42 + ? excluded) ===')
    cur.execute('SELECT dominant_type, COUNT(*) FROM public.fact_paper_id_card GROUP BY dominant_type ORDER BY 2 DESC;')
    total = 0
    rows = cur.fetchall()
    total = sum(r[1] for r in rows)
    for d, c in rows:
        print(f'  {d}: {c:>11,}  ({100*c/total:5.2f}%)')

    print('\n=== PaperCard: 5 sample (random) ===')
    cur.execute('SELECT paper_id, pmid, language, year, dominant_type, type_confidence FROM public.fact_paper_id_card TABLESAMPLE BERNOULLI (0.001) LIMIT 5;')
    for row in cur.fetchall():
        print(f'  {row}')

    print('\n=== GhostCard: top 5 indegree ===')
    cur.execute('SELECT paper_id, indegree, outdegree, pagerank, n_corpus_citers, language_inferred FROM public.dim_ghost_paper ORDER BY indegree DESC NULLS LAST LIMIT 5;')
    for row in cur.fetchall():
        print(f'  {row}')

    print('\n=== JSONB sanity (topic_profile.theme_ids length avg) ===')
    cur.execute('SELECT AVG(jsonb_array_length(topic_profile->\'theme_ids\'))::numeric(6,2) FROM public.fact_paper_id_card TABLESAMPLE BERNOULLI (0.1);')
    print(f'  avg theme_ids per paper (sample 0.1%): {cur.fetchone()[0]}')

    print('\n=== PaperCard v1.2: is_suspicious flag (envanter hedef ~20,677 = %0.083) ===')
    cur.execute('SELECT is_suspicious, COUNT(*) FROM public.fact_paper_id_card GROUP BY is_suspicious ORDER BY 2 DESC;')
    for v, c in cur.fetchall():
        print(f'  is_suspicious={v}: {c:>11,}')

    print('\n=== GhostCard v1.2: triage_priority dağılımı (envanter hedef high=75,159 / medium=143,877 / low=540,123 / none=31,096,278) ===')
    cur.execute('SELECT triage_priority, COUNT(*) FROM public.dim_ghost_paper GROUP BY triage_priority ORDER BY 2 DESC;')
    for v, c in cur.fetchall():
        print(f'  {v:8s}: {c:>11,}')

    print('\n=== Disk usage ===')
    cur.execute('''SELECT relname, pg_size_pretty(pg_total_relation_size(oid)) FROM pg_class
                   WHERE relname IN ('fact_paper_id_card','dim_ghost_paper') AND relkind='r';''')
    for n, s in cur.fetchall():
        print(f'  {n}: {s}')

print('\n✓ Faz 1 (PaperCard + GhostCard) verify complete')